In [1]:
# SPDX-FileCopyrightText: Copyright (c) 2025-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

# 🕵️ Rewriting Legal Documents

Rewriting legal text (TAB dataset) with a domain-specific privacy goal
and custom entity labels tailored for legal proceedings.

#### 📚 What you'll learn

- Define domain-specific entity labels for legal text (case numbers, court names, etc.)
- Configure rewrite mode with legal-specific privacy goals
- Preview and run on court decision documents
- Triage flagged records with `needs_human_review`

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
- Import `Detect` (for custom entity labels), `Rewrite`, and its config classes.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.
- `configure_logging()` controls verbosity -- switch to `LoggingConfig.debug()` when troubleshooting.

In [2]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [3]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, Detect, LoggingConfig, Rewrite, configure_logging
from anonymizer.config.rewrite import EvaluationCriteria, PrivacyGoal, RiskTolerance

configure_logging(LoggingConfig.default())

In [4]:
anonymizer = Anonymizer()

[18:59:47] [INFO] 🔧 Anonymizer initialized with 3 model configs


[18:59:47] [INFO]   |-- 🔎 detector:  gliner-pii-detector


[18:59:47] [INFO]   |-- ✅ validator: gpt-oss-120b


[18:59:47] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Input data

- [TAB (Text Anonymization Benchmark)](https://github.com/NorskRegnesentral/text-anonymization-benchmark)
  legal documents -- court decisions containing names, dates, case numbers, and other legal identifiers.
- `LEGAL_ENTITY_LABELS` defines the domain-specific entity types to detect.
  This replaces the default label set with one tailored to legal text.

In [5]:
LEGAL_ENTITY_LABELS = [
    "first_name",
    "last_name",
    "court_name",
    "organization_name",
    "company_name",
    "prison_detention_facility",
    "street_address",
    "city",
    "state",
    "country",
    "date",
    "date_time",
    "time",
    "date_of_birth",
    "age",
    "email",
    "phone_number",
    "ssn",
    "unique_id",
    "legal_role",
    "case_number",
    "application_number",
    "monetary_amount",
    "sentence_duration",
    "nationality",
]

input_data = AnonymizerInput(
    source="../data/TAB_legal_sample25.csv",
    text_column="text",
    data_summary="Legal court decisions containing personal identifiers, case numbers, and institutional references",
)

## 🎛️ Configure

- `Detect(entity_labels=...)` overrides the default entity set with legal-specific labels.
- `PrivacyGoal` tells the rewriter what to **protect** (identifiers, case numbers,
  institutional references) and what to **preserve** (legal reasoning, statutory references,
  ruling structure).

In [6]:
config = AnonymizerConfig(
    detect=Detect(
        entity_labels=LEGAL_ENTITY_LABELS,
    ),
    rewrite=Rewrite(
        privacy_goal=PrivacyGoal(
            protect="All personal identifiers, case numbers, court names, and institutional references that could identify parties",
            preserve="Legal reasoning, procedural facts, statutory references, and the structure of the ruling",
        ),
        evaluation=EvaluationCriteria(
            risk_tolerance=RiskTolerance.strict,
            max_repair_iterations=3,
        ),
    ),
)

## 👁️ Preview

- Preview on a few records to check that legal entities are detected
  and the rewrite preserves the ruling's structure.

In [7]:
preview = anonymizer.preview(
    config=config,
    data=input_data,
    num_records=3,
)

preview.display_record(0)

[18:59:47] [INFO] 📂 Loaded 25 records from ../data/TAB_legal_sample25.csv (column: 'text')


[18:59:47] [INFO] detection labels in scope: ['age', 'application_number', 'case_number', 'city', 'company_name', 'country', 'court_name', 'date', 'date_of_birth', 'date_time', 'email', 'first_name', 'last_name', 'legal_role', 'monetary_amount', 'nationality', 'organization_name', 'phone_number', 'prison_detention_facility', 'sentence_duration', 'ssn', 'state', 'street_address', 'time', 'unique_id']


[18:59:47] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records


[18:59:47] [INFO] 🔍 Running entity detection on 3 records


[19:01:51] [INFO]   |-- 📋 Detection complete — 137 entities found across 3 records (0 failed) [124.4s]


...

[19:09:58] [INFO] Evaluate-repair loop iteration 1: 1/3 rows need repair


[19:11:18] [INFO] Evaluate-repair loop iteration 2: 1/3 rows need repair


[19:13:41] [INFO]   |-- 📋 Rewrite complete (0 failed) [709.8s]


[19:13:41] [INFO] 🎉 Pipeline complete — 3 records processed, 0 total failures


Entity,Label,Sensitivity,Protection
74463/01,application_number,medium,replace
Republic of Turkey,country,low,left_as_is
Turkish,nationality,low,left_as_is
Feriştah,first_name,high,replace
Bahçeyaka,last_name,high,replace
8 June 2001,date,medium,generalize
Kuloğlu,last_name,high,replace
lawyer,legal_role,low,left_as_is
Aydın,city,low,left_as_is
Turkish,nationality,low,left_as_is


In [8]:
preview.display_record(1)

Entity,Label,Sensitivity,Protection
29360/06,application_number,high,replace
Republic of Poland,country,low,left_as_is
Polish,nationality,low,left_as_is
Teresa,first_name,high,replace
Jerzak,last_name,high,replace
3 July 2006,date,medium,generalize
Agent,legal_role,low,left_as_is
Wołąsiewicz,last_name,high,replace
Ministry of Foreign Affairs,organization_name,low,left_as_is
25 September 2007,date,medium,generalize


## 🚀 Full run

- `result.dataframe` has user-facing columns: rewritten text, scores, and the review flag.

In [9]:
result = anonymizer.run(config=config, data=input_data)

result.dataframe.head()

[19:13:41] [INFO] 📂 Loaded 25 records from ../data/TAB_legal_sample25.csv (column: 'text')


[19:13:41] [INFO] detection labels in scope: ['age', 'application_number', 'case_number', 'city', 'company_name', 'country', 'court_name', 'date', 'date_of_birth', 'date_time', 'email', 'first_name', 'last_name', 'legal_role', 'monetary_amount', 'nationality', 'organization_name', 'phone_number', 'prison_detention_facility', 'sentence_duration', 'ssn', 'state', 'street_address', 'time', 'unique_id']


[19:13:41] [INFO] 🔍 Running entity detection on 25 records


[19:24:10] [INFO]   |-- 📋 Detection complete — 1307 entities found across 25 records (0 failed) [629.1s]


[19:24:10] [INFO]   |-- labels: date=419, court_name=238, legal_role=181, organization_name=84, last_name=83, first_name=59, nationality=53, city=48, country=43, application_number=26, date_of_birth=25, prison_detention_facility=18, sentence_duration=13, monetary_amount=10, state=3, case_number=1, age=1, time=1, company_name=1


...

[20:03:06] [INFO] Evaluate-repair loop iteration 2: 8/24 rows need repair


/Users/lramaswamy/Documents/github/Anonymizer/src/anonymizer/engine/rewrite/rewrite_workflow.py:86: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(list(parts), ignore_index=True)
[20:05:55] [INFO]   |-- 📋 Rewrite complete (1 failed) [2504.6s]


[20:05:55] [WARNING] 1 record(s) failed during pipeline processing.


[20:05:55] [INFO] 🎉 Pipeline complete — 25 records processed, 1 total failures


,text,text_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
0,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,1.0,0.0,False,False
1,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.972727,1.6,True,True
2,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,1.0,1.8,False,False
3,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.990909,1.0,True,True
4,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.984615,1.6,True,True


In [10]:
result.dataframe[["text_rewritten", "utility_score", "leakage_mass", "needs_human_review"]].head()

,text_rewritten,utility_score,leakage_mass,needs_human_review
0,PROCEDURE The case originated in an applicati...,1.0,0.0,False
1,PROCEDURE The case originated in an applicati...,0.972727,1.6,True
2,PROCEDURE The case originated in an applicati...,1.0,1.8,False
3,PROCEDURE The case originated in an applicati...,0.990909,1.0,True
4,PROCEDURE The case originated in an applicati...,0.984615,1.6,True


## 🚩 Filter by review flag

- Records where automated metrics exceed thresholds are flagged for manual review.
- Use this to prioritize human attention on the records that need it most.

In [11]:
df = result.dataframe
flagged = df[df["needs_human_review"] == True]  # noqa: E712
print(f"{len(flagged)} of {len(df)} records flagged for human review")
flagged.head()

6 of 24 records flagged for human review


,text,text_rewritten,utility_score,leakage_mass,any_high_leaked,needs_human_review
1,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.972727,1.6,True,True
3,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.990909,1.0,True,True
4,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.984615,1.6,True,True
7,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.79,4.0,True,True
12,PROCEDURE The case originated in an applicati...,The case originated in an application (no. 315...,0.285714,0.3,False,True


## ⏭️ Next steps

- **[🔍 Inspecting Detected Entities](02_inspecting_detected_entities.ipynb)** --
  debug what the detection pipeline found before rewriting.
- **Try it on your own data!** Swap in your CSV, define entity labels for your
  domain, and set a `PrivacyGoal` that fits -- you've got all the building blocks.